# Diseño experimental

## Marco General

El diseño del benchmark integra un estudio de ablación implícito. La conformación del conjunto combina 10 modelos de Deep Learning, cuatro baselines econométricos de referencia (GARCH, EGARCH, HAR y HAR-WTD) y dos pronósticos triviales. Los pronósticos triviales corresponden al modelo ingenuo (*Naive*), que asigna a la predicción $\hat{y}_t$ el valor observado en el instante anterior $y_{t-1}$, y al *Naive* estacional, que utiliza el valor del mismo día de la semana anterior.

Esta estructura completa permite aislar el efecto de componentes arquitectónicos específicos sin requerir experimentos adicionales. La comparación entre un LSTM y un BiLSTM aísla el efecto de la bidireccionalidad. De igual forma, la inclusión de modelos como WTD-HAR-GRU o GARCH-LSTM permite medir el aporte de la ingeniería de características y la integración econométrica. El rendimiento diferencial entre todas estas arquitecturas proporciona la evidencia necesaria para identificar qué componente aporta la mejora predictiva frente al pronóstico trivial.

Para garantizar la paridad de información y prevenir el sesgo de anticipación (*look-ahead bias*), el diseño experimental restringe el insumo a los retornos logarítmicos históricos del crudo Brent. En cualquier instante $t$, todos los modelos procesan exclusivamente la información disponible hasta $t-1$. Esta información se estructura como una matriz de rezagos de retornos logarítmicos, denotada como $\mathbf{X}_t = [r_{t-L}, \dots, r_{t-1}]$. 

Formalmente, el problema se define como una tarea de regresión aplicada al pronóstico de series de tiempo (*1-step-ahead*). El objetivo es encontrar una función $f$ perteneciente a un espacio de hipótesis $\mathcal{H}$ (definido por cada arquitectura) tal que:

$$
y_t = f(\mathbf{X}_t; \theta) + \varepsilon_t
$$

Donde $y_t$ es el valor de la volatilidad realizada de Rogers-Satchell ($RS_t$) en el instante $t$, $\theta$ representa los parámetros entrenables del modelo y $\varepsilon_t$ es el término de error. Los parámetros $\theta$ se optimizan minimizando la función de pérdida establecida para el benchmark.

La selección del estimador RS responde a tres razones metodológicas. Primero, procesa los precios de apertura, máximo, mínimo y cierre (OHLC), incorporando la trayectoria intradía del activo. Segundo, supera en eficiencia estadística a los retornos al cuadrado, los cuales presentan una alta varianza y omiten la información intradía. Tercero, ignora la deriva del precio, una propiedad adecuada para materias primas con tendencias marcadas. Las predicciones de todos los modelos se contrastan contra esta serie diaria para calcular las funciones de pérdida y las métricas de evaluación.

El estudio adopta la pérdida QLIKE como métrica principal, en lugar de métricas tradicionales como el RMSE o el MSE. Las métricas tradicionales presentan una estructura simétrica que penaliza por igual la sobreestimación y la subestimación de la volatilidad. Esta simetría no refleja el riesgo financiero real, donde subestimar la volatilidad implica una mayor pérdida económica. 

La volatilidad no es observable directamente y se estima mediante una proxy ruidosa. Investigaciones previas demuestran que el uso de MSE o RMSE con proxies ruidosas altera el ranking de los modelos evaluados. La función QLIKE mantiene la consistencia del ranking bajo ruido de medición y penaliza con mayor severidad la subestimación.

:::{admonition} Información: Función de Pérdida QLIKE
:class: info

La función de pérdida QLIKE (por sus siglas en inglés *Quasi-Likelihood*) se emplea para evaluar pronósticos de volatilidad. Dados $\sigma_t^2$ como el proxy de volatilidad realizada y $\hat{\sigma}_t^2$ como la varianza condicional pronosticada, la función se define como:

$$
\text{QLIKE}(\sigma_t^2, \hat{\sigma}_t^2) = \frac{1}{T} \sum_{t=1}^{T} \left( \frac{\sigma_t^2}{\hat{\sigma}_t^2} - \ln\left(\frac{\sigma_t^2}{\hat{\sigma}_t^2}\right) - 1 \right)
$$

La función QLIKE presenta una estructura asimétrica, otorgando una mayor penalización a la subestimación de la volatilidad frente a su sobreestimación. De acuerdo con Patton (2011), el uso de esta función garantiza que el ordenamiento de los modelos no se altere por errores de medición en la proxy de volatilidad.
:::

### Evaluación Fuera de la Muestra y Métricas de Rendimiento

Para la fase de prueba (Out-of-Sample), el protocolo de evaluación aplica de forma estricta el mismo esquema de predicción descrito en la fase de entrenamiento. El conjunto de prueba se recorre secuencialmente día a día, donde en cada paso $t$ el modelo predice la volatilidad de $t+1$. Los modelos econométricos reestiman sus parámetros mediante MLE u OLS cada 5 días hábiles, mientras que los modelos de Deep Learning actualizan sus pesos secuencialmente mediante *Warm-Start* y *Early Stopping*.

La evaluación del rendimiento se fundamenta en tres métricas primarias: la pérdida QLIKE, el Pinball Loss y el Continuous Ranked Probability Score (CRPS). Este trío de métricas constituye el núcleo de la evaluación dado que capturan la asimetría inherente a la volatilidad y la calibración de la distribución predictiva. 

:::{admonition} Información: Función de Pérdida Pinball y CRPS
:class: info

La función Pinball evalúa la precisión de un pronóstico probabilístico en un cuantil $\tau$ específico. Dados $y_t$ como la volatilidad realizada y $\hat{q}_{\tau,t}$ como el cuantil pronosticado, se define como:

$$
L_\tau(y_t, \hat{q}_{\tau,t}) = \max \left( \tau (y_t - \hat{q}_{\tau,t}), (\tau - 1)(y_t - \hat{q}_{\tau,t}) \right)
$$

El Continuous Ranked Probability Score (CRPS) integra esta pérdida sobre todos los cuantiles $\tau \in (0, 1)$. Dado que la integral analítica es inviable para distribuciones de colas pesadas, se aplica la cuadratura de Gauss-Legendre con 20 nodos. Este método numérico es exacto para polinomios de grado 39. Aunque 20 nodos pueden parecer una elección excesiva para la mayoría de funciones, esta configuración garantiza que el error de truncamiento resultante sea del orden de $10^{-15}$ (precisión de máquina), lo que elimina el sesgo computacional en la evaluación.
:::

De forma complementaria, el benchmark reporta las métricas de error estándar MSE, RMSE y MAE, así como las métricas de heterocedasticidad ajustada HMSE y HMAE. Estas métricas secundarias no intervienen en la selección del modelo, pero se incluyen para garantizar la comparación directa con el corpus de literatura revisado y evaluar la magnitud del error puntual.

Para verificar que los modelos capturaron la estructura de dependencia temporal y no dejaron información sin explotar, se aplica una batería de pruebas sobre los residuos estandarizados de cada configuración final. Se establece un nivel de significancia de $\alpha = 0.05$ y se aplica la corrección de Bonferroni para controlar la tasa de error por familia (FWER). La prueba de Ljung-Box evalúa la autocorrelación lineal; la prueba ARCH-LM verifica la presencia de heterocedasticidad condicional; las pruebas de Jarque-Bera y Lilliefors examinan la normalidad; y la prueba BDS detecta dependencias no lineales ocultas. Los resultados se reportan para cada modelo, indicando el estadístico, el p-valor ajustado y la conclusión correspondiente.

Para diagnosticar la presencia de sobreajuste, se define un umbral basado en la diferencia absoluta entre las métricas de prueba y entrenamiento: $\Delta_{abs} = M_{test} - M_{train}$. Un incremento del QLIKE fuera de la muestra superior a 0.05 indica una pérdida de capacidad predictiva. Para las métricas de error (MSE, RMSE, MAE, HMSE, HMAE) y probabilísticas (CRPS, Pinball), se establece una tolerancia del 5% respecto al valor de entrenamiento. Un valor de $\Delta_{abs}$ que supere estos márgenes evidencia sobreajuste y falta de generalización.

### Comparación Estadística y Robustez

Para el análisis de significancia estadística y la comparación formal entre modelos, se aplican el test de Diebold-Mariano y el Model Confidence Set (MCS). Se establece un nivel de significancia de $\alpha = 0.05$ para todas las pruebas de hipótesis. Los intervalos de confianza y los $p$-valores se calculan mediante remuestreo bootstrap con $B = 2000$ iteraciones para preservar la estructura de autocorrelación de los errores. Dado que la significancia estadística no implica relevancia práctica, la magnitud de la diferencia se evalúa mediante el tamaño del efecto utilizando la Delta de Cliff $\left(\delta_{Cliff}\right)$. Se establece un umbral de 0.33 para considerar una diferencia como prácticamente relevante (efecto moderado).

La validez de un modelo superior en promedio requiere verificar su comportamiento bajo condiciones adversas. Para ello, se evalúa la robustez de las configuraciones óptimas inyectando ruido gaussiano aditivo a los retornos y simulando valores faltantes (*missing at random*). Un modelo robusto mantiene una pérdida QLIKE estable bajo estas perturbaciones, demostrando que no depende de una condición específica del mercado.

Asimismo, se aplica la técnica de Valores de Shapley aditivos (SHAP, por sus siglas en inglés *SHapley Additive exPlanations*) e Integrated Gradients a los tres mejores modelos. El objetivo de la interpretabilidad es identificar qué rezagos o componentes exógenos (como la varianza GARCH o los modos Wavelet) influyen con mayor peso en la predicción, verificando la coherencia con el conocimiento del dominio financiero y descartando el aprendizaje de atajos (*shortcut learning*).

:::{admonition} Información: Valores de Shapley (SHAP)
:class: info

Los valores de Shapley, adaptados al aprendizaje automático bajo el marco SHAP (*SHapley Additive exPlanations*), asignan a cada atributo una contribución marginal a la predicción del modelo. Dado un conjunto de atributos $F$, el valor de Shapley $\phi_i$ para la característica $i$ promedia su efecto marginal sobre todas las posibles coaliciones de características:

$$
\phi_i = \sum_{S \subseteq F \setminus \{i\}} \frac{|S|! (|F| - |S| - 1)!}{|F|!} \left[ f(S \cup \{i\}) - f(S) \right]
$$

Donde $S$ es un subconjunto de atributos que no contiene a $i$, y $f(S)$ representa la predicción del modelo condicionada únicamente a las características en $S$. El método garantiza consistencia local, permitiendo descomponer la predicción en la suma de las contribuciones individuales de cada variable de entrada.
:::

Finalmente, el análisis de errores segmenta la pérdida QLIKE por regímenes de volatilidad (alta versus baja) y por periodos de crisis estructural (e.g., la crisis sanitaria de 2020 o la crisis energética de 2022). Este análisis permite identificar si un modelo, a pesar de tener un buen rendimiento promedio, falla sistemáticamente en los eventos extremos, los cuales representan el mayor interés para la gestión de riesgo y la toma de decisiones.

## Preguntas de Investigación e Hipótesis

El benchmark se rige por tres preguntas de investigación (RQ) contrastables mediante las pruebas estadísticas definidas en el protocolo.

RQ1: ¿Las arquitecturas evaluadas capturan de forma efectiva las dependencias lineales y no lineales inherentes a la serie de volatilidad del Brent, dejando residuos libres de estructura (ruido blanco)?
*   $H_0$: Los residuos de los modelos presentan estructura lineal (autocorrelación o efectos ARCH) o no lineal (dependencias ocultas), lo que indica una captura incompleta de la dinámica de la volatilidad.
*   $H_1$: Los residuos de los modelos se comportan como ruido blanco, sin autocorrelación lineal, sin efectos ARCH y sin estructura no lineal remanente.
*   Prueba: Test de Ljung-Box para dependencia lineal, test ARCH-LM para heterocedasticidad condicional y test BDS para dependencia no lineal.

RQ2: ¿Las arquitecturas de Deep Learning superan a los modelos econométricos tradicionales en el pronóstico de volatilidad del Brent bajo un protocolo libre de data leakage?
*   $H_0$: No existe diferencia significativa en la pérdida QLIKE entre los modelos de Deep Learning y los baselines econométricos.
*   $H_1$: Los modelos de Deep Learning reducen significativamente la pérdida QLIKE frente a los baselines.
*   Prueba: Test de Diebold-Mariano (con corrección HLN) y Model Confidence Set (MCS).

RQ3: ¿Las arquitecturas híbridas (con descomposición o integración econométrica) superan a las arquitecturas puras de Deep Learning?
*   $H_0$: El rango promedio de rendimiento de los modelos híbridos no difiere del de los modelos puros.
*   $H_1$: Los modelos híbridos presentan un mejor rango promedio significativo.
*   Prueba: Test de Friedman con post-hoc de Nemenyi y diagrama de diferencia crítica.